### 1. Initialize the Model
Set up the API key and initialize the LLM (e.g., Groq).

In [1]:
from langchain.chat_models import init_chat_model
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

### 2. Define Structured Schema with Pydantic
Create Pydantic models to strictly define the expected output format.

In [13]:
from pydantic import BaseModel, Field

class genres(BaseModel):
    name: str = Field(..., description="The name of the genre")
    description: str = Field(..., description="The description of the genre")
    
class games(BaseModel):
    name: str = Field(description="The name of the game")
    description: str = Field(description="The description of the game")
    price: float = Field(description="The price of the game")
    studio: str = Field(description="The studio that developed the game")
    release_date: str = Field(description="The release date of the game")
    genre: list[genres] = Field(description="The genre's of the game")

### 3. Generate Structured Output
Use .with_structured_output() to force the model to respond in the defined format.

In [15]:
structured_model = model.with_structured_output(games, include_raw=False) # include_raw to get raw response from the model

response = structured_model.invoke("Tell me about Elden Ring")
print(response)

name='Elden Ring' description='Elden Ring is an action role-playing game set in a vast open world known as the Lands Between. Players embark on a journey to reclaim the Elden Ring, a powerful artifact that has been shattered, restoring balance to the realm. The game features challenging combat, exploration, and a richly detailed dark fantasy setting.' price=59.99 studio='Bandai Namco Entertainment (developed by FromSoftware)' release_date='2022-02-25' genre=[genres(name='Action RPG', description='Combines real-time action gameplay with role-playing elements, including character progression and inventory management.'), genres(name='Dark Fantasy', description='Incorporates a stylized, grim, and often ominous fantasy atmosphere with complex storytelling.')]


### TypedDict
To avoid input validation we can use typedict in same way as pydantic

In [23]:
from typing_extensions import TypedDict , Annotated

class genres_1(TypedDict):
    name: str = Field(..., description="The name of the genre")
    description: str = Field(..., description="The description of the genre")
    
class games_1(TypedDict):
    name: str = Field(description="The name of the game")
    description: str = Field(description="The description of the game")
    price: float = Field(description="The price of the game")
    studio: str = Field(description="The studio that developed the game")
    release_date: str = Field(description="The release date of the game")
    genre: list[genres] = Field(description="The genre's of the game")

In [24]:
structured_model = model.with_structured_output(games_1) 

response = structured_model.invoke("Tell me about Elden Ring")
response

{'description': 'Elden Ring is an action role-playing game set in a vast, interconnected world known as the Lands Between. Players assume the role of a Tarnished, a warrior exiled in the Lands Between, seeking to reclaim the Elden Ring and become the Elden Lord. The game features challenging combat, exploration, and a richly detailed dark fantasy narrative.',
 'genre': [{'description': 'A role-playing game that emphasizes action elements such as combat, exploration, and real-time interactions.',
   'name': 'Action RPG'}],
 'name': 'Elden Ring',
 'price': 59.99,
 'release_date': '2022-02-25',
 'studio': 'FromSoftware'}

### Dataclass


In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class cs2_event_info:
    """ Information of cs2 event (Tournament)"""
    event: str
    start_date:str
    end_date:str    
    winner:str
    prize_pool:str


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=cs2_event_info
)

response = agent.invoke({"messages": [{"role": "user", "content": "Extract the cs2 event info from : CS2 Austin major was won by Team vitality on 2nd of march with the prize pool of 250000$, the event started at 1 feb"}]})

response

{'messages': [HumanMessage(content='Extract the cs2 event info from : CS2 Austin major was won by Team vitality on 2nd of march with the prize pool of 250000$, the event started at 1 feb', additional_kwargs={}, response_metadata={}, id='ae106d62-fe74-4d99-be69-c5c97cad3ae9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s tackle this user query. They want me to extract the CS2 event info from the given text. The user provided a specific example about the Austin major.\n\nFirst, I need to identify the key pieces of information required by the function. The function parameters are event, start_date, end_date, winner, and prize_pool. The user mentioned that the event was won by Team Vitality on the 2nd of March, with a prize pool of $250,000, and it started on February 1st.\n\nWait, the user didn\'t mention the end date explicitly. The example text says the event started on 1st Feb and ended on 2nd March? Or is the end date not provided? Let me check again. 

In [28]:
response['structured_response']

cs2_event_info(event='CS2 Austin Major', start_date='2023-02-01', end_date='2023-03-02', winner='Team Vitality', prize_pool='$250,000')

In [29]:
response

{'messages': [HumanMessage(content='Extract the cs2 event info from : CS2 Austin major was won by Team vitality on 2nd of march with the prize pool of 250000$, the event started at 1 feb', additional_kwargs={}, response_metadata={}, id='ae106d62-fe74-4d99-be69-c5c97cad3ae9'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s tackle this user query. They want me to extract the CS2 event info from the given text. The user provided a specific example about the Austin major.\n\nFirst, I need to identify the key pieces of information required by the function. The function parameters are event, start_date, end_date, winner, and prize_pool. The user mentioned that the event was won by Team Vitality on the 2nd of March, with a prize pool of $250,000, and it started on February 1st.\n\nWait, the user didn\'t mention the end date explicitly. The example text says the event started on 1st Feb and ended on 2nd March? Or is the end date not provided? Let me check again. 